<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/solved/10_exgaussian_default_priors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 10 — Ex-Gaussian with default priors

Fit the same distributional ex-Gaussian structure using Bambi defaults. This parallels the chapter’s comparison between carefully chosen priors and software defaults.

## Setup

This course pins PyMC, modular ArviZ, and Bambi for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1" \
    "bambi==0.21.0"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("Bambi:", bmb.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

In [ ]:
def diagnostic_overview(idata):
    """Compact sampling diagnostics used throughout the notebook sequence."""
    divergences = int(idata["sample_stats"]["diverging"].sum().item())
    rhat = idata["posterior"].azstats.rhat().to_array()
    ess = idata["posterior"].azstats.ess(method="bulk").to_array()
    print("Divergences:", divergences)
    print("Worst R-hat:", float(rhat.max(skipna=True).item()))
    print("Smallest bulk ESS:", float(ess.min(skipna=True).item()))


def hdi_bounds(draws, prob):
    hdi = azs.hdi(draws, prob=prob)
    return (
        hdi.sel(ci_bound="lower").to_numpy(),
        hdi.sel(ci_bound="upper").to_numpy(),
    )


def expected_response_draws(prediction, family):
    """Expected response on the reaction-time scale for the families used here."""
    posterior = prediction["posterior"]
    if family in {"gaussian", "t"}:
        return posterior["mu"]
    if family == "lognormal":
        return np.exp(posterior["mu"] + 0.5 * posterior["sigma"] ** 2)
    if family == "exgaussian":
        return posterior["mu"] + posterior["nu"]
    raise ValueError(f"Unsupported family: {family}")


def plot_population_fit(model, idata, family="gaussian", title="Population-average relationship"):
    """Posterior mean relationship, excluding participant-specific deviations."""
    grid = pd.DataFrame({"Days": np.linspace(0, 7, 100)})
    pred = model.predict(
        idata,
        kind="response_params",
        data=grid,
        inplace=False,
        include_group_specific=False,
        random_seed=RANDOM_SEED,
    )
    draws = expected_response_draws(pred, family)
    mean = draws.mean(("chain", "draw")).to_numpy()
    lo50, hi50 = hdi_bounds(draws, 0.50)
    lo90, hi90 = hdi_bounds(draws, 0.90)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(sleep["Days"], sleep["Reaction"], s=16, color="black", alpha=0.32)
    ax.fill_between(grid["Days"], lo90, hi90, alpha=0.25, label="90% HDI")
    ax.fill_between(grid["Days"], lo50, hi50, alpha=0.45, label="50% HDI")
    ax.plot(grid["Days"], mean, color="C1", lw=2, label="mean")
    ax.set(xlabel="Days of sleep deprivation", ylabel="Reaction time (ms)", title=title)
    ax.legend(frameon=False)
    plt.show()


def plot_subject_fits(model, idata, family="gaussian", title="Participant trajectories"):
    """Posterior mean relationship for each observed participant."""
    subjects = sleep["Subject"].drop_duplicates().tolist()
    day_grid = np.linspace(0, 7, 50)
    grid = pd.DataFrame(
        [(subject, day) for subject in subjects for day in day_grid],
        columns=["Subject", "Days"],
    )
    pred = model.predict(
        idata,
        kind="response_params",
        data=grid,
        inplace=False,
        include_group_specific=True,
        random_seed=RANDOM_SEED,
    )
    draws = expected_response_draws(pred, family)
    mean = draws.mean(("chain", "draw")).to_numpy().reshape(len(subjects), len(day_grid))
    lo90, hi90 = hdi_bounds(draws, 0.90)
    lo90 = lo90.reshape(len(subjects), len(day_grid))
    hi90 = hi90.reshape(len(subjects), len(day_grid))

    fig, axes = plt.subplots(3, 6, figsize=(12, 7), sharex=True, sharey=True)
    for i, (ax, subject) in enumerate(zip(axes.ravel(), subjects)):
        observed = sleep[sleep["Subject"] == subject]
        ax.scatter(observed["Days"], observed["Reaction"], s=15, color="black", zorder=3)
        ax.fill_between(day_grid, lo90[i], hi90[i], alpha=0.20)
        ax.plot(day_grid, mean[i], color="C1", lw=1.5)
        ax.set_title(f"Subject {subject}", fontsize=9)
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    plt.show()

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every Bambi model in this sequence uses `center_predictors=False`. The `Intercept` prior is therefore a prior on baseline reaction time rather than reaction time at the average deprivation day.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

# 10.1 Software-default priors

What prior assumptions does Bambi supply for this ex-Gaussian model when we do not specify them ourselves?

The formula is unchanged. We deliberately omit `priors=` so the notebook shows exactly what Bambi supplies by default. `center_predictors=False` remains explicit because baseline at `Days = 0` is scientifically meaningful.

In [ ]:
formula = bmb.Formula(
    "Reaction ~ Days + (1 + Days | Subject)",
    "sigma ~ 1 + (1 | Subject)",
    "nu ~ 1",
)

model = bmb.Model(
    formula, sleep, family="exgaussian", categorical="Subject",
    center_predictors=False,
)
model

In [ ]:
prior = model.prior_predictive(draws=500, random_seed=RANDOM_SEED)
azp.plot_ppc_dist(
    prior,
    group="prior_predictive",
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 10.2 Posterior changes under defaults

Which posterior features change when the custom ex-Gaussian priors are replaced by Bambi’s defaults?

In [ ]:
idata = model.fit(draws=1000, tune=2000, chains=4, target_accept=0.95, random_seed=RANDOM_SEED)
diagnostic_overview(idata)

In [ ]:
azs.summary(idata, ci_prob=0.90, ci_kind="hdi", round_to=2)

In [ ]:
plot_population_fit(model, idata, family="exgaussian")

In [ ]:
plot_subject_fits(model, idata, family="exgaussian")

# 10.3 Predictive robustness to defaults

Do Bambi’s default priors materially change the model’s predictive behavior?

In [ ]:
model.predict(
    idata,
    kind="response",
    inplace=True,
    random_seed=RANDOM_SEED,
)

azp.plot_ppc_dist(
    idata,
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 10.4 Parameter-level sensitivity

Which parameters appear prior-sensitive even when the overall predictions remain similar?

Compare the common deprivation effect, the tail parameter, group-level scales, sampling diagnostics, and predictive distribution with notebook 9. Similar posterior results for some parameters do not imply that every weakly identified parameter is prior-insensitive.